In [ ]:
# import libraries

import json

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
# Initialize LabelEncoder
le = LabelEncoder()

Add a new tab to the dashboard app that Marius previously demonstrated, titled "Transaction Analysis." This tab will begin with a brief explanation or table summarizing the dataset's structure and metadata to help users understand the features. A model XGBoost is used to prediction frauds. Texts will show the confusion matrix of the model, and a ROC-AUC plot is shown as fixed image. Feature importances are also shown as a bar plot after the ROC-AUC plot. Following this, a SHAP plot will be presented to visualize feature importance at the transaction level. Each point on the SHAP plot represents a single transaction; when a user hovers over a point, a pop-up will display key information such as the transaction ID, user ID, and card ID. Additionally, users will be able to input a specific card ID into a search field. Based on the input, the dashboard will display a filtered view of transactions related to that card, highlighting those predicted as fraudulent in red for easy identification.


In [ ]:
# Load data
# Download latest version
import kagglehub
path = kagglehub.dataset_download("computingvictor/transactions-fraud-datasets")

print("Path to dataset files:", path)
# Load CSV files
card_data = pd.read_csv("/Users/m/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1/cards_data.csv")
transactions = pd.read_csv("/Users/m/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1/transactions_data.csv")
users_data = pd.read_csv("/Users/m/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1/users_data.csv")
# Load Json files
with open("/Users/m/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1/mcc_codes.json", "r") as f:
    mcc_codes = json.load(f)
with open("/Users/m/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1/train_fraud_labels.json", "r") as f:
    train_fraud_labels = json.load(f)

In [ ]:
# Performing data preparation on transactions_data

# convert the date column to date type object
transactions_data['date'] = pd.to_datetime(transactions_data['date'])

# convert date 
transactions_data['year'] = transactions_data['date'].dt.year
transactions_data['month'] = transactions_data['date'].dt.month
transactions_data['day'] = transactions_data['date'].dt.day
transactions_data['hour'] = transactions_data['date'].dt.hour
transactions_data['minute'] = transactions_data['date'].dt.minute

# getting the target dict from train_fraud_labels
target_dict = train_fraud_labels['target']
target_dict = {int(k):v for k,v in target_dict.items()}

# map the fraud_labels to transactions_data
transactions_data['target'] = transactions_data['id'].map(target_dict)

# convert the amount to numerical value
transactions_data['amount'] = transactions_data['amount'].replace('[\$,]', '', regex=True).astype('float')
# encode on the 'use_chip' column
transactions_data['use_chip_encoded'] = le.fit_transform(transactions_data['use_chip'])

# Fit and transform the 'merchant_id' column
transactions_data['merchant_id_encoded'] = le.fit_transform(transactions_data['merchant_id'])

# encode merchant city
transactions_data['merchant_city_encoded'] = le.fit_transform(transactions_data['merchant_city'])

# encode merchant state
transactions_data['merchant_state_encoded'] = le.fit_transform(transactions_data['merchant_state'])

# encode zip value
transactions_data['zip_encoded'] = le.fit_transform(transactions_data['zip'])

# encode mcc column
transactions_data['mcc_encoded'] = le.fit_transform(transactions_data['mcc'])
# encode errors column
transactions_data['errors_encoded'] = le.fit_transform(transactions_data['errors'])

# create a seperate df to hold mapping between merchant_id and merchant_id_encoded before dropping merchant_id
transactions_data_encode_mapping = transactions_data[['use_chip', 'use_chip_encoded', 'merchant_id', 'merchant_id_encoded', 'merchant_city', 'merchant_city_encoded', 'merchant_state', 'merchant_state_encoded', 'zip', 'zip_encoded', 'mcc', 'mcc_encoded', 'errors', 'errors_encoded']]

# Drop original columns which have been encoded
transactions_data = transactions_data.drop(columns=['use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'date'])

# update target with binary values 
binary_target = {'Yes': 1, 'No': 0}
transactions_data['target'] = transactions_data['target'].map(binary_target).fillna(-1).astype('int')

In [ ]:
# Data preprocessing for cards_data

# Label encoding for 'card_brand', 'card_type' , 'has_chip' 'card_on_dark_web', 

# encoding for card_brand
cards_data['card_brand_encoded'] = le.fit_transform(cards_data['card_brand'])

# encoding for card_type
cards_data['card_type_encoded'] = le.fit_transform(cards_data['card_type'])

# encoding for card_number
cards_data['card_number_encoded'] = le.fit_transform(cards_data['card_number'])

# convert expires column to datetime format
cards_data['expires_formatted'] = pd.to_datetime(cards_data['expires'])

# convert expires column to year and month
cards_data['expires_year'] = cards_data['expires_formatted'].dt.year
cards_data['expires_month'] = cards_data['expires_formatted'].dt.month
# do label encoding for cvv, incase if some cvv are easier to guess
cards_data['cvv_encoded'] = le.fit_transform(cards_data['cvv'])

# label encoding for 'has_chip' column
cards_data['has_chip_encoded'] = le.fit_transform(cards_data['has_chip'])

# convert credit_limit column to float
cards_data['credit_limit'] = cards_data['credit_limit'].replace('[\$,]', '', regex=True).astype('float')

# convert expires column to datetime format
cards_data['acct_open_date_formatted'] = pd.to_datetime(cards_data['acct_open_date'])

# convert expires column to year and month
cards_data['acct_open_year'] = cards_data['acct_open_date_formatted'].dt.year
cards_data['acct_open_month'] = cards_data['acct_open_date_formatted'].dt.month

# label encoding for card_on_dark_web
cards_data['card_on_dark_web_encoded'] = le.fit_transform(cards_data['card_on_dark_web'])
# keeping the encoded mapping in a seperate df
cards_data_encode_mapping = cards_data[['card_brand', 'card_brand_encoded', 'card_type', 'card_type_encoded', 'card_number', 'card_number_encoded', 'cvv', 'cvv_encoded', 'has_chip', 'has_chip_encoded', 'card_on_dark_web', 'card_on_dark_web_encoded']]

# drop the original columns which have been encoded
cards_data = cards_data.drop(columns=['card_brand', 'card_type', 'card_number', 'expires', 'cvv', 'has_chip', 'acct_open_date', 'card_on_dark_web', 'expires_formatted', 'acct_open_date_formatted'])

In [ ]:
# Data preparation for users_data

# keeping the current_age, retirement_age, birth_year, birth_month columns as it is.
# they may not be relevant for the models but will still keep adding them.

# label encoding for gender column
users_data['gender_encoded'] = le.fit_transform(users_data['gender'])

# convert per_capita_income, yearly_income, total_debt to numeric
users_data['per_capita_income'] = users_data['per_capita_income'].replace('[\$,]', '', regex=True).astype('float')
users_data['yearly_income'] = users_data['yearly_income'].replace('[\$,]', '', regex=True).astype('float')
users_data['total_debt'] = users_data['total_debt'].replace('[\$,]', '', regex=True).astype('float')

# drop columns address and gender
users_data = users_data.drop(columns=['address', 'gender'])

In [ ]:
# merging transactions_data and cards_data

transactions_cards_merged = transactions_data.merge(cards_data, left_on='card_id', right_on='id', how='left')
# merging users_data to transactions_data

merged_df = transactions_cards_merged.merge(users_data, left_on='client_id_x', right_on='id', how='left')
# remove all the unnecessary columns

columns_to_drop = ['id_x', 'client_id_x', 'card_id', 'id_y', 'client_id_y', 'id', 'card_on_dark_web_encoded', 'current_age']

merged_df = merged_df.drop(columns=columns_to_drop)


In [ ]:
# split the data into train & test split

from sklearn.model_selection import train_test_split

# drop the unlabelled data
labeled_df = merged_df[merged_df['target'] != -1]

X = labeled_df.drop(columns=['target'])
y = labeled_df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

XGBoost model to predict fraud

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(scale_pos_weight=99.85/0.15, random_state=42)  # Adjust weight for fraud class
xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)

# Evaluate model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
print("Confusion Matrix:\n", cm)

Evaluate XGBoost

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_test, y_pred)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')  # diagonal line
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

Feature Importance (feature importance bar plot, shap beeworms plot)

In [ ]:
# --- XGBoost Feature Importance ---
feature_importance_xgb = pd.Series(xgb.feature_importances_, index=X.columns)
feature_importance_xgb = feature_importance_xgb.sort_values(ascending=False)

# Plot XGBoost top 20
feature_importance_xgb[:20].plot(kind='bar', ax=axes[1])
axes[1].set_title("Top 20 Features: XGBoost")

In [ ]:
import shap
import pandas as pd
import matplotlib.pyplot as plt

# Ensure test set is a DataFrame with column names
X_test_df = pd.DataFrame(X_test, columns=X.columns)

# Create a SHAP explainer
explainer = shap.Explainer(xgb, X_train)

# Calculate SHAP values for all samples
shap_values = explainer(X_test_df, check_additivity=False)
shap.summary_plot(shap_values, X_test_df, plot_type="dot", max_display=20)

Percard Visualization

In [ ]:
# Load data
# Download latest version
path = kagglehub.dataset_download("computingvictor/transactions-fraud-datasets")

print("Path to dataset files:", path)
# Load CSV files
card_data = pd.read_csv("/Users/m/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1/cards_data.csv")
transactions = pd.read_csv("/Users/m/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1/transactions_data.csv")
users_data = pd.read_csv("/Users/m/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1/users_data.csv")
# Load Json files
with open("/Users/m/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1/mcc_codes.json", "r") as f:
    mcc_codes = json.load(f)
with open("/Users/m/.cache/kagglehub/datasets/computingvictor/transactions-fraud-datasets/versions/1/train_fraud_labels.json", "r") as f:
    train_fraud_labels = json.load(f)


In [ ]:
labels_dict = train_fraud_labels["target"] # Extract the dictionary from JSON
# Map the label using dictionary to transactions data
transactions["label"] = transactions["id"].astype(str).map(labels_dict)
# Test set: where 'label' is NaN
test_set = transactions[transactions['label'].isna()].copy()

# Training set: where 'label' is NOT NaN
train_set = transactions[transactions['label'].notna()].copy()


print('percentage of test: ', round(test_set.shape[0] / (train_set.shape[0] + train_set.shape[0]), 4))

# Display the counts
print(f"Training Set: {train_set.shape[0]} rows")
print(f"Test Set: {test_set.shape[0]} rows")

In [ ]:
# Group by 'card_id' and count the number of transactions
card_transaction_counts = train_set.groupby('card_id').size().reset_index(name='transaction_count')

# Group by 'card_id' and count the number of 'Yes' labels (anomalies)
card_label_anomalies = train_set[train_set['label'] == 'Yes'].groupby('card_id').size().reset_index(name='label_anormality_count')

# Merge the two DataFrames on 'card_id'
card_transaction_counts = pd.merge(
    card_transaction_counts, 
    card_label_anomalies, 
    on='card_id', 
    how='left'
)

# Fill NaN values in 'label_anormality_count' with 0 (no anomalies for those cards)
card_transaction_counts['label_anormality_count'] = card_transaction_counts['label_anormality_count'].fillna(0).astype(int)

# Sort by transaction count / label_anormality count
card_transaction_counts = card_transaction_counts.sort_values(by='transaction_count', ascending=False).reset_index(drop=True)
card_anormal_transaction_counts = card_transaction_counts.sort_values(by='label_anormality_count', ascending=False).reset_index(drop=True)

# Display the result
print(card_transaction_counts.head())  # Display top cards
print(card_anormal_transaction_counts.head())  # Display top anormal cards
print(card_transaction_counts.tail(10))  # Display bottom cards

In [ ]:
train_set['amount'] = train_set['amount'].replace({'\$': '', ',': ''}, regex=True).astype(float)
train_set['date'] = pd.to_datetime(train_set['date'])

In [ ]:
# Sort by mcc and date for correct moving window calculation
train_set_transformed = train_set.copy().sort_values(by=['mcc', 'date'])

# Define a custom moving window parameter
moving_window = 10 # Customize this parameter as needed
ema_span = 5  # Customize EMA sensitivity

# Calculate moving average and moving standard deviation per mcc
train_set_transformed['moving_mean'] = train_set_transformed.groupby('mcc')['amount'].transform(lambda x: x.rolling(window=moving_window, min_periods=1).mean())
train_set_transformed['moving_std'] = train_set_transformed.groupby('mcc')['amount'].transform(lambda x: x.rolling(window=moving_window, min_periods=1).std(ddof=0))

# Calculate exponential moving average (EMA)
train_set_transformed['ema'] = train_set_transformed.groupby('mcc')['amount'].transform(
    lambda x: x.ewm(span=ema_span, adjust=False).mean()
)

# Centralized and standardized transactions
train_set_transformed['centralized_transaction'] = train_set_transformed['amount'] - train_set_transformed['moving_mean']
train_set_transformed['standardized_transaction'] = train_set_transformed['centralized_transaction'] / train_set_transformed['moving_std'].replace(0, 1)  # Avoid division by zero

# Display the result
pd.set_option('display.float_format', lambda x: '%.2f' % x)  # Format floats for better readability
print(train_set_transformed.head())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_card_transaction_with_fraud(transactions, card_id, xgb_model, amount_col="amount", feature_cols=None):
    """
    Plots a transaction feature over time for a specific card ID, highlighting predicted frauds in red.

    Parameters:
    transactions (pd.DataFrame): The full transaction dataset.
    card_id (int or str): The card ID to filter.
    xgb_model: The trained XGBoost model used for fraud prediction.
    amount_col (str): The column name to plot (e.g., "amount", "ema", "moving_mean").
    feature_cols (list): The list of feature column names used for prediction.

    Returns:
    None (Displays a plot)
    """
    
    # Filter transactions for the specific card
    card_data = transactions[transactions["card_id"] == card_id].copy()
    
    if card_data.empty:
        print(f"No transactions found for Card ID {card_id}.")
        return

    # Ensure date column is datetime
    card_data["date"] = pd.to_datetime(card_data["date"])

    # Check amount column
    if amount_col not in card_data.columns:
        print(f"Column '{amount_col}' not found in the dataset.")
        return

    # Drop missing or zero values in the amount column
    card_data = card_data.dropna(subset=[amount_col])
    if card_data[amount_col].sum() == 0:
        print(f"All values for '{amount_col}' in Card ID {card_id} are zero.")
        return

    # Sort by date
    card_data = card_data.sort_values(by="date")

    # Predict fraud using the model if feature columns are given
    if feature_cols is not None:
        if not all(col in card_data.columns for col in feature_cols):
            print("Some feature columns are missing in the card_data.")
            return
        card_data["predicted_fraud"] = xgb_model.predict(card_data[feature_cols])
    else:
        print("feature_cols must be specified for prediction.")
        return

    # Plot
    plt.figure(figsize=(12, 5))

    # Normal points
    nonfraud = card_data[card_data["predicted_fraud"] == 0]
    plt.plot(nonfraud["date"], nonfraud[amount_col], marker='o', linestyle='-', markersize=5, alpha=0.7, label="Normal")

    # Fraud points
    fraud = card_data[card_data["predicted_fraud"] == 1]
    plt.plot(fraud["date"], fraud[amount_col], marker='o', linestyle='', color='red', markersize=7, label="Predicted Fraud")

    # Title and labels
    plt.title(f"{amount_col} Over Time for Card ID {card_id}")
    plt.xlabel("Date")
    plt.ylabel(amount_col.replace("_", " ").title())
    plt.grid(True)
    plt.legend()

    # Show plot
    plt.show()


In [ ]:
# Example configuration
engineered_transactions = ["moving_mean", "ema", "centralized_transaction", "standardized_transaction"]

# Display cards with the fewest transactions - Adjust this that allows user to input a card id to get the corresponding result
print(card_transaction_counts.tail())
least_transaction_cards = list(card_transaction_counts.tail()['card_id'])
print(least_transaction_cards)

# Loop through each card and each engineered transaction feature
for card_id in least_transaction_cards:
    for amount_col in engineered_transactions:
        plot_card_transaction_with_fraud(
            transactions=train_set_transformed,
            card_id=card_id,
            xgb_model=xgb,
            amount_col=amount_col,
            feature_cols=feature_cols  # replace with your actual feature column list
        )
